In [15]:
import re
import string
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
import nltk


In [16]:

nltk.download('punkt_tab')    # for tokenization
nltk.download('stopwords')    # for stopwords removal


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [17]:

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

## **Clean and normalize text for ML.**
Makes text more uniform, reduces noise, and simplifies vocabulary for model training.

In [18]:
def preprocess(text):
    text = text.lower()  # Lowercase
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    text = re.sub(r'\d+', '', text)  # Remove numbers
    tokens = text.split()

    # negation_words = {"not", "no", "nor", "n't"}
    # tokens = [word for word in tokens if word not in stop_words or word in negation_words]

    tokens = [word for word in tokens if word not in stop_words]  # Remove stopwords

    tokens = [stemmer.stem(word) for word in tokens]  # Stemming
    return ' '.join(tokens)

In [19]:
from google.colab import drive
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:


# csv_path = '/content/drive/My Drive/NLP/Restaurant_Reviews.tsv'


# .tsv to csv
# with open(csv_path, 'r', newline='') as tsvfile, open('/content/drive/My Drive/NLP/Restaurant_Reviews.csv', 'w', newline='') as csvfile:
#     tsv_reader = csv.reader(tsvfile, delimiter='\t')
#     csv_writer = csv.writer(csvfile, delimiter=',')
#     for row in tsv_reader:
#         csv_writer.writerow(row)


In [21]:
import csv

# df = pd.DataFrame(data)
df = pd.read_csv("/content/drive/My Drive/NLP/bbc_data.csv")
df

,data,labels
0,Musicians to tackle US red tape Musicians gro...,entertainment
1,"U2s desire to be number one U2, who have won ...",entertainment
2,Rocker Doherty in on-stage fight Rock singer ...,entertainment
3,Snicket tops US box office chart The film ada...,entertainment
4,"Oceans Twelve raids box office Oceans Twelve,...",entertainment
...,...,...
2220,Warning over Windows Word files Writing a Mic...,tech
2221,Fast lifts rise into record books Two high-sp...,tech
2222,Nintendo adds media playing to DS Nintendo is...,tech
2223,Fast moving phone viruses appear Security fir...,tech


In [22]:
df['clean_data'] = df['data'].apply(preprocess)
df

,data,labels,clean_data
0,Musicians to tackle US red tape Musicians gro...,entertainment,musician tackl us red tape musician group tack...
1,"U2s desire to be number one U2, who have won ...",entertainment,us desir number one u three prestigi grammi aw...
2,Rocker Doherty in on-stage fight Rock singer ...,entertainment,rocker doherti onstag fight rock singer pete d...
3,Snicket tops US box office chart The film ada...,entertainment,snicket top us box offic chart film adapt lemo...
4,"Oceans Twelve raids box office Oceans Twelve,...",entertainment,ocean twelv raid box offic ocean twelv crime c...
...,...,...,...
2220,Warning over Windows Word files Writing a Mic...,tech,warn window word file write microsoft word doc...
2221,Fast lifts rise into record books Two high-sp...,tech,fast lift rise record book two highspe lift wo...
2222,Nintendo adds media playing to DS Nintendo is...,tech,nintendo add media play ds nintendo releas ada...
2223,Fast moving phone viruses appear Security fir...,tech,fast move phone virus appear secur firm warn s...


# **TF-IDF vectorisation**
converts text into numerical vectors

In [23]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['clean_data'])
y = df['labels']

In [24]:
y.unique()

array(['entertainment', 'business', 'sport', 'politics', 'tech'],
      dtype=object)

In [25]:
y

,labels
0,entertainment
1,entertainment
2,entertainment
3,entertainment
4,entertainment
...,...
2220,tech
2221,tech
2222,tech
2223,tech


In [26]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 319136 stored elements and shape (2225, 21496)>

# **Model Training**

In [27]:
def train_and_evaluate_model(model, X_train, y_train, X_test, y_test, name="Model"):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"\n====== {name} ======")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))
    return model



In [28]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


In [29]:
# Logistic Regression
lr_model = LogisticRegression()
lr_model = train_and_evaluate_model(lr_model, X_train, y_train, X_test, y_test, name="Logistic Regression")

# Naive Bayes
nb_model = MultinomialNB()
nb_model = train_and_evaluate_model(nb_model, X_train, y_train, X_test, y_test, name="Naive Bayes")

# Support Vector Classifier
svc_model = SVC(kernel='linear')  # Use 'linear' for text classification
svc_model = train_and_evaluate_model(svc_model, X_train, y_train, X_test, y_test, name="Support Vector Classifier")



====== Logistic Regression ======
Accuracy: 0.9790419161676647
Classification Report:
                precision    recall  f1-score   support

     business       0.96      0.99      0.98       163
entertainment       0.97      0.97      0.97       120
     politics       0.96      0.98      0.97       112
        sport       1.00      0.99      1.00       148
         tech       1.00      0.96      0.98       125

     accuracy                           0.98       668
    macro avg       0.98      0.98      0.98       668
 weighted avg       0.98      0.98      0.98       668


====== Naive Bayes ======
Accuracy: 0.9670658682634731
Classification Report:
                precision    recall  f1-score   support

     business       0.96      0.98      0.97       163
entertainment       0.99      0.93      0.96       120
     politics       0.91      0.99      0.95       112
        sport       0.99      0.99      0.99       148
         tech       0.98      0.93      0.95       125

  

In [30]:
def predict_news_category(text, model):
    clean = preprocess(text)
    vec = vectorizer.transform([clean])
    result = model.predict(vec)[0]
    return result


In [33]:
example_text_1 = "Apple unveils new iPhone with camera upgrade"
example_text_2 = "Amazon's shares surged after record-breaking profits."

for model, name in zip([lr_model, nb_model, svc_model], ["Logistic Regression", "Naive Bayes", "SVC"]):
    print(f"\n{name} Prediction 1: {predict_news_category(example_text_1, model)}")
    print(f"{name} Prediction 2: {predict_news_category(example_text_2, model)}")



Logistic Regression Prediction 1: tech
Logistic Regression Prediction 2: business

Naive Bayes Prediction 1: tech
Naive Bayes Prediction 2: business

SVC Prediction 1: tech
SVC Prediction 2: business
